# Gatekeeper, giai đoạn 0 (§37), phần A: Kaggle có chấm được task SWE-smith với cùng điểm không?

**Trước khi chạy** (khung Settings bên phải): **Internet: On**, **Accelerator: None**.

Chạy lần lượt ô 1 đến ô 4. **Mỗi ô tự đủ**: nếu kernel bị khởi động lại, chỉ cần chạy lại ô 1 rồi tiếp tục ở ô đang dở.
Ô 3 và ô 4 chạy việc nặng trong một tiến trình riêng và in tiến độ ngay khi có.
Không ô nào in khóa hay thông tin tài khoản. Kết quả ở `/kaggle/working/`: `k0a.json`, `k0b_kaggle.jsonl`.

In [ ]:
# Ô 1 — cài đặt (khoảng 2–3 phút; chạy lại an toàn)
%env UDOCKER_DIR=/kaggle/working/udocker
!pip -q install "swesmith==0.0.9" "swebench==4.1.0" udocker unidiff 2>&1 | tail -2
!udocker --allow-root install 2>&1 | tail -1
!docker info > /dev/null 2>&1 && echo "Docker daemon: CO" || echo "Docker daemon: KHONG"
!df -h /kaggle/working ~ / | tail -3

In [ ]:
# ruff: noqa: F811 (every cell redefines the same helpers on purpose: a cell must run alone)
import os
import subprocess
import sys

W = "/kaggle/working"
os.environ["UDOCKER_DIR"] = f"{W}/udocker"  # 20 GB here; the default ~/.udocker may be small
CODE = next(
    (
        f"{W}/{d}"
        for d in ("gatekeeper/kaggle/scripts", "gatekeeper/scripts")
        if os.path.exists(f"{W}/{d}/udocker_shim.py")
    ),
    None,
)


def run(cmd):
    """Run a step in its own process; stream its output; a crash cannot kill this kernel."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    print("exit code:", p.wait(), flush=True)


# Ô 2 — mã và dữ liệu (repo công khai; không cần đăng nhập)
REPO = "dinhthithuthuyduphong-bot/gatekeeper"
BRANCH = "claude/gatekeeper-13-research-nxytoh"  # Kaggle work lives here until it runs
os.chdir(W)
if not os.path.isdir("gatekeeper"):
    url = f"https://github.com/{REPO}"
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url], check=True)
else:
    subprocess.run(["git", "-C", "gatekeeper", "pull", "-q"], check=False)
RAW = f"https://raw.githubusercontent.com/{REPO}"
files = {
    "swesmith_pallets__jinja.jsonl": "data-snapshots/data/instances/swesmith_pallets__jinja.jsonl",
    "f1_probe.json": "trial/results/f1_probe.json",
    "f1_skill_probe.json": "trial/results/f1_skill_probe.json",
}
os.makedirs("k0data", exist_ok=True)
for name, path in files.items():
    dest = f"k0data/{name}"
    if not os.path.exists(dest):
        subprocess.run(["curl", "-sSfL", "-o", dest, f"{RAW}/{path}"], check=True)
    print(name, os.path.getsize(dest), "byte")
head = subprocess.run(
    ["git", "-C", "gatekeeper", "log", "--oneline", "-1"], capture_output=True, text=True
)
print(head.stdout)
CODE = f"{W}/gatekeeper/kaggle/scripts"
print("code:", CODE, os.path.exists(f"{CODE}/k0a_probe.py"))

In [ ]:
import os
import subprocess
import sys

W = "/kaggle/working"
os.environ["UDOCKER_DIR"] = f"{W}/udocker"  # 20 GB here; the default ~/.udocker may be small
CODE = next(
    (
        f"{W}/{d}"
        for d in ("gatekeeper/kaggle/scripts", "gatekeeper/scripts")
        if os.path.exists(f"{W}/{d}/udocker_shim.py")
    ),
    None,
)


def run(cmd):
    """Run a step in its own process; stream its output; a crash cannot kill this kernel."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    print("exit code:", p.wait(), flush=True)


# Ô 3 — K0a: image của task có chạy được không; thử các cách chạy của udocker (5–15 phút)
run(
    [
        sys.executable,
        f"{CODE}/k0a_probe.py",
        "--image",
        "swebench/swesmith.x86_64.pallets_1776_jinja.ada0a9a6",
        "--out-dir",
        W,
    ]
)

In [ ]:
import os
import subprocess
import sys

W = "/kaggle/working"
os.environ["UDOCKER_DIR"] = f"{W}/udocker"  # 20 GB here; the default ~/.udocker may be small
CODE = next(
    (
        f"{W}/{d}"
        for d in ("gatekeeper/kaggle/scripts", "gatekeeper/scripts")
        if os.path.exists(f"{W}/{d}/udocker_shim.py")
    ),
    None,
)


def run(cmd):
    """Run a step in its own process; stream its output; a crash cannot kill this kernel."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    print("exit code:", p.wait(), flush=True)


# Ô 4 — K0b: chấm lại các bản vá đã lưu, so với điểm đã ghi dưới Docker
# (30–60 phút; chạy lại được: các lượt đã chấm được bỏ qua)
import shutil  # noqa: E402

mode = open(f"{W}/k0a_mode.txt").read().strip() if os.path.exists(f"{W}/k0a_mode.txt") else ""
if not mode:
    raise SystemExit("Ô 3 chưa tìm được cách chạy nào (k0a_mode.txt rỗng): gửi k0a.json cho Claude")
os.environ["UDOCKER_EXECMODE"] = mode
print("udocker execution mode:", mode)
os.makedirs(f"{W}/k0run", exist_ok=True)
os.chdir(f"{W}/k0run")  # SWE-smith writes its logs under ./logs/run_evaluation
D = f"{W}/k0data"
run(
    [
        sys.executable,
        f"{CODE}/k0_stage0.py",
        "--rows",
        f"{D}/swesmith_pallets__jinja.jsonl",
        "--results",
        f"{D}/f1_probe.json",
        f"{D}/f1_skill_probe.json",
        "--controls",
        "20",
        "--out",
        f"{W}/k0b_kaggle.jsonl",
    ]
)
shutil.rmtree(f"{W}/k0run/logs", ignore_errors=True)

## Sau khi chạy xong

Gửi lại `/kaggle/working/k0a.json` và `/kaggle/working/k0b_kaggle.jsonl` (khung Output, nút Download), cùng phần in ra của ô 3 và vài dòng cuối của ô 4.

Phần B (K0c, K0d: gọi model bằng hạn mức AI) chạy ở notebook tạo từ https://www.kaggle.com/benchmarks/tasks/new, bằng ô `kaggle/notebooks/k0cd_benchmarks_cell.py`.